# Arabic Sentiment Classification with AraBERT
Fine-tune `aubmindlab/bert-base-arabertv02` to classify Arabic e-commerce reviews (SHEIN) as **Negative / Neutral / Positive**.

**Pipeline:** Setup → Data → Tokenize → Train → Evaluate → Save & Predict

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece seaborn

In [ ]:
import os, re, random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

@dataclass
class Config:
    SEED: int = 42
    MODEL_NAME: str = "aubmindlab/bert-base-arabertv02"
    DATA_PATH: str = "Ruqiya/Arabic_Reviews_of_SHEIN"   # Hugging Face dataset
    LOCAL_CSV: str = "data/shein_arabic_reviews.csv"    # fallback (columns: raw_text/review, label/sentiment)
    MAX_LENGTH: int = 128
    TRAIN_BATCH_SIZE: int = 16
    EVAL_BATCH_SIZE: int = 32
    LEARNING_RATE: float = 2e-5             # pretrained encoder
    CLASSIFIER_LEARNING_RATE: float = 1e-4  # new classification head
    WEIGHT_DECAY: float = 0.01
    NUM_EPOCHS: int = 4
    WARMUP_RATIO: float = 0.1
    MAX_GRAD_NORM: float = 1.0
    PATIENCE: int = 2                       # early stopping on validation macro F1
    OUTPUT_DIR: str = "outputs/final_model"

cfg = Config()
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(CPU will be slow)")

## 2. Data
Stars are mapped to sentiment (1–2 → Negative, 3 → Neutral, 4–5 → Positive). Text is normalized conservatively: diacritics, tatweel, hamza variants and letter elongation are cleaned, while **emojis, `!`/`?` and negation words are kept** because they carry sentiment. Duplicates are removed *before* an 80/10/10 stratified split to prevent leakage.

In [ ]:
def normalize_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\u064B-\u0652\u0670\u0640]", "", text)       # tashkeel + tatweel
    text = re.sub(r"[إأآٱ]", "ا", text)                              # hamza/alif variants
    text = re.sub(r"ى\b", "ي", text)
    text = re.sub(r"ة\b", "ه", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                      # collapse elongation
    text = re.sub(r"([\U00010000-\U0010ffff])", r" \1 ", text)     # isolate emojis
    text = re.sub(r"([!؟?])", r" \1 ", text)
    return re.sub(r"\s+", " ", text).strip()

def stars_to_sentiment(v):
    v = int(float(v))
    return 0 if v <= 2 else (1 if v == 3 else 2)

# Load: Hugging Face first, local CSV as fallback
try:
    from datasets import load_dataset
    df = load_dataset(cfg.DATA_PATH, split="train").to_pandas()
except Exception as e:
    print("HF load failed, using local CSV:", e)
    df = pd.read_csv(cfg.LOCAL_CSV)

text_col = "raw_text" if "raw_text" in df.columns else "review"
label_col = "label" if "label" in df.columns else "sentiment"
df = df.dropna(subset=[text_col, label_col]).copy()
df["label"] = df[label_col].map(stars_to_sentiment)
df["text"] = df[text_col].map(normalize_arabic)
df = df[df["text"].str.len() > 0].drop_duplicates(subset="text")[["text", "label"]]

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=cfg.SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=cfg.SEED, stratify=temp_df["label"])
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:<5}: {len(d):>6} | class counts: {d['label'].value_counts().sort_index().to_dict()}")

## 3. Tokenization & DataLoaders (dynamic padding)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, frame):
        self.texts, self.labels = frame["text"].tolist(), frame["label"].tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return self.texts[i], self.labels[i]

def collate(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), truncation=True, max_length=cfg.MAX_LENGTH, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels)
    return enc

train_loader = DataLoader(ReviewDataset(train_df), batch_size=cfg.TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(ReviewDataset(val_df), batch_size=cfg.EVAL_BATCH_SIZE, collate_fn=collate)
test_loader = DataLoader(ReviewDataset(test_df), batch_size=cfg.EVAL_BATCH_SIZE, collate_fn=collate)

## 4. Train AraBERT
Full fine-tuning with AdamW (higher LR for the new classifier head), linear warmup/decay, mixed precision, and **inverse-frequency class weights** to handle the Neutral class imbalance. The best epoch is chosen by validation macro F1.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.MODEL_NAME, num_labels=3, classifier_dropout=0.2, id2label=ID2LABEL, label2id=LABEL2ID
).to(device)

# Class-weighted loss (weights from the training split only)
counts = np.bincount(train_df["label"], minlength=3)
weights = torch.tensor(len(train_df) / (3 * counts), dtype=torch.float, device=device)
weights = weights / weights.mean()
loss_fn = nn.CrossEntropyLoss(weight=weights)

# Optimizer: different LRs for encoder vs classifier head, no decay on bias/LayerNorm
no_decay = ("bias", "LayerNorm.weight")
groups = []
for is_head in (False, True):
    for decay in (True, False):
        params = [p for n, p in model.named_parameters()
                  if n.startswith("classifier") == is_head and (not any(nd in n for nd in no_decay)) == decay]
        groups.append({"params": params,
                       "lr": cfg.CLASSIFIER_LEARNING_RATE if is_head else cfg.LEARNING_RATE,
                       "weight_decay": cfg.WEIGHT_DECAY if decay else 0.0})
optimizer = torch.optim.AdamW(groups)
total_steps = len(train_loader) * cfg.NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(cfg.WARMUP_RATIO * total_steps), total_steps)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, targets, total_loss = [], [], 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device.type, enabled=use_amp):
            logits = model(**batch).logits
        total_loss += loss_fn(logits.float(), labels).item() * len(labels)
        preds += logits.argmax(-1).cpu().tolist()
        targets += labels.cpu().tolist()
    return {"loss": total_loss / len(targets), "macro_f1": f1_score(targets, preds, average="macro"),
            "accuracy": accuracy_score(targets, preds), "preds": preds, "targets": targets}

best_f1, bad_epochs, best_state = -1.0, 0, None
for epoch in range(1, cfg.NUM_EPOCHS + 1):
    model.train()
    running = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=use_amp):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
    val = evaluate(val_loader)
    print(f"Epoch {epoch}/{cfg.NUM_EPOCHS} | train loss {running/len(train_loader):.4f} | "
          f"val loss {val['loss']:.4f} | val macro F1 {val['macro_f1']:.4f} | val acc {val['accuracy']:.4f}")
    if val["macro_f1"] > best_f1:
        best_f1, bad_epochs = val["macro_f1"], 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= cfg.PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(best_state)
print(f"Best validation macro F1: {best_f1:.4f}")

## 5. Final Test Evaluation
The test set is evaluated once, with the best checkpoint.

In [ ]:
test = evaluate(test_loader)
print(f"Test macro F1: {test['macro_f1']:.4f} | accuracy: {test['accuracy']:.4f} | target >= 0.80 met: {test['macro_f1'] >= 0.80}\n")
names = list(ID2LABEL.values())
print(classification_report(test["targets"], test["preds"], target_names=names, digits=4))

cm = confusion_matrix(test["targets"], test["preds"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Test Confusion Matrix"); plt.show()

## 6. Save Model & Inference

In [ ]:
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
model.save_pretrained(cfg.OUTPUT_DIR)
tokenizer.save_pretrained(cfg.OUTPUT_DIR)
print("Saved to", cfg.OUTPUT_DIR)

@torch.no_grad()
def predict_sentiment(text: str) -> dict:
    model.eval()
    enc = tokenizer(normalize_arabic(text), truncation=True, max_length=cfg.MAX_LENGTH, return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits.float(), dim=-1)[0].cpu().numpy()
    return {"label": ID2LABEL[int(probs.argmax())], "confidence": float(probs.max()),
            "probabilities": {ID2LABEL[i]: float(p) for i, p in enumerate(probs)}}

for t in ["المنتج رائع جدا وأنصح به 😍", "مش كويس خالص وما عجبني", "المنتج عادي"]:
    print(t, "->", predict_sentiment(t))